# Workspaces

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

A **workspace** is a per-project sandbox: it has its own Python virtual environment, its own list of installed packages, its own notebooks and scripts, and its own plugin overrides. SciQLop activates the workspace's venv on startup, so anything you install with `%install` stays local to that workspace and survives restarts.

This separation matters because each science task — fitting a magnetopause model, processing MSA telemetry, running a wave analysis pipeline — typically needs different libraries and SciQLop plugins. Keeping them in separate workspaces prevents conflicts.

**What you'll learn**
- Inspect the current workspace.
- Install packages into the workspace and have them persist across restarts.
- Check which plugins are enabled/disabled for the current workspace.
- Browse and copy examples into the workspace.
- See how notebooks carry their environment, and pick the SciQLop version a workspace runs.

**Prerequisites** — [SciQLop GUI tour](1-SciQLopGUIDiscovery.ipynb).

**Next** — [Speasy first steps](../Speasy/2-SpeasyFirstSteps.ipynb).

## 1. Workspace status


`%workspace status` prints the workspace's name, on-disk path, recorded dependencies and plugin overrides — a one-stop summary.


In [ ]:
%workspace status

## 2. Listing dependencies

See which packages are recorded in the workspace manifest.

In [ ]:
%workspace deps

## 3. Installing packages

`%install <pkg>` (or the longer `%workspace install <pkg>`) calls `uv pip install` inside the workspace venv **and** records the package in `workspace.sciqlop`, so it is reinstalled automatically on the next launch.

We'll install `astropy` — a small, harmless dependency you might want for some space-physics work. After the install, `%workspace deps` should list it.


In [ ]:
%install astropy


You can do the same from Python with the public API — this is what scripts and
the embedded AI agent use:

```python
from SciQLop.user_api.packages import install_packages
install_packages("astropy")
# -> {"ok": True, "installed": [...], "already_present": [...], "error": ""}
```

AI agents add dependencies through the gated `sciqlop_install_package` tool,
which routes through this same path — so anything they install is recorded in
`workspace.sciqlop` and persists. Never rely on a bare `pip install`: it is not
recorded and is wiped when the venv is rebuilt.

In [ ]:
from SciQLop.user_api.packages import install_packages
install_packages("astropy")

In [ ]:
# Confirm astropy is now recorded in the manifest
%workspace deps


## 4. Plugin overrides

See which plugins are enabled or disabled specifically for this workspace.

In [ ]:
%workspace plugins

## 5. Browsing and adding examples

SciQLop ships with example notebooks. You can list them and copy any example into your workspace.

In [ ]:
# List available examples
%workspace examples

In [ ]:
# Copy an example bundle into the current workspace.
# (The example name must match one returned by %workspace examples above —
#  uncomment the line and edit if you want to actually copy one.)
# %workspace add-example "<example name>"


## 6. Notebooks remember their environment

Every time you save a notebook, SciQLop records the SciQLop version and the workspace's packages (with their installed versions) in the notebook metadata, under `sciqlop`:

```json
"metadata": {"sciqlop": {"version": "0.14.0", "dependencies": ["astropy==7.0.1"]}}
```

Local packages and plugin folders are left out. When you open a notebook that needs packages this workspace lacks, or a newer SciQLop, SciQLop offers to install them here or to create a new workspace that matches the notebook. So a notebook you share brings its environment along.


## 7. Choosing the SciQLop version

Each workspace runs its own SciQLop. Its card on the welcome page shows which one: a release such as `0.13.1`, or `main`. A pinned release older than the newest one on PyPI is shown in orange.

To change it, click the card, pick a version under **SciQLop version**, and press **Install**. **main (development)** follows the latest code. If it is the workspace you are running, SciQLop offers to restart.


## 8. Getting help

In [ ]:
%workspace help

## Where do workspaces live?

Each workspace is a folder holding a `workspace.sciqlop` manifest (TOML) and a `.venv/` directory. `%workspace status` prints the folder of the current one.

New workspaces are created in the `workspaces` folder of SciQLop's data folder:

| OS | Default location |
|----|------------------|
| Linux | `~/.local/share/sciqlop/workspaces/` |
| macOS | `~/Library/Application Support/sciqlop/workspaces/` |
| Windows | `%LOCALAPPDATA%\LPP\sciqlop\workspaces\` |

To create them somewhere else, change **Settings › Workspaces › Workspaces Dir**.

## Reference

| Command | Purpose |
|---------|--------|
| `%workspace status` | Name, path, summary |
| `%workspace deps` | List recorded dependencies |
| `%workspace install <pkg>` | Install + record (alias for `%install`) |
| `%workspace plugins` | List plugin overrides |
| `%workspace examples` | List available examples |
| `%workspace add-example <name>` | Copy an example into the workspace |
| `%install <pkg> [pkg2 ...]` | Install packages into the workspace |
